# Structural Intervention Distance


In [1]:
"""Setup helpers for the bnm example notebooks.

These notebooks are intentionally self-contained: they use small local
utilities (random DAG generator, simple synthetic-data sampler, basic
PC-CPDAG construction) so a reader can run them with only ``bnm`` and
its optional ``[viz]`` extra installed. In production, prefer
``dagsampler`` for DAG construction and synthetic data and ``cbcd`` for
the PC algorithm and CPDAG output.
"""

import bnm
import numpy as np
import pandas as pd
import random
from collections import deque

# Configure plotly so figures render in any notebook viewer (JupyterLab,
# classic Notebook, nbviewer, GitHub's static renderer) instead of only
# emitting a vnd.plotly.v1+json mimetype.
import plotly.io as pio
pio.renderers.default = "notebook_connected+plotly_mimetype"


# ---- DAG / data generation (would normally come from dagsampler) ----

def random_dag(n_nodes=40, edge_prob=0.1, seed=None):
    """Return a bnm.GraphLike random DAG with X_1..X_n nodes."""
    rng = random.Random(seed)
    names = tuple(f"X_{i+1}" for i in range(n_nodes))
    topo = list(range(n_nodes))
    rng.shuffle(topo)
    endpoints = np.zeros((n_nodes, n_nodes), dtype=np.int8)
    for i in range(n_nodes):
        for j in range(i + 1, n_nodes):
            if rng.random() < edge_prob:
                src, dst = topo[i], topo[j]
                endpoints[src, dst] = bnm.EndpointMark.ARROW
                endpoints[dst, src] = bnm.EndpointMark.TAIL
    return bnm.to_graphlike(endpoints, var_names=names)


def generate_data(g, n_samples=1000, stdev=1.0, seed=None):
    """Linear-Gaussian SCM sampling. Returns a pandas DataFrame
    with columns matching g.var_names."""
    rng = np.random.default_rng(seed)
    n = g.n_vars
    names = list(g.var_names) if g.var_names else [str(i) for i in range(n)]
    arr = np.array(g.endpoints)
    children: list[list[int]] = [[] for _ in range(n)]
    in_deg = [0] * n
    for i in range(n):
        for j in range(n):
            if (
                arr[i, j] == bnm.EndpointMark.ARROW
                and arr[j, i] == bnm.EndpointMark.TAIL
            ):
                children[i].append(j)
                in_deg[j] += 1
    queue = deque(v for v in range(n) if in_deg[v] == 0)
    topo: list[int] = []
    while queue:
        v = queue.popleft()
        topo.append(v)
        for c in children[v]:
            in_deg[c] -= 1
            if in_deg[c] == 0:
                queue.append(c)
    data = pd.DataFrame(index=range(n_samples), columns=names, dtype=float)
    for v in topo:
        parents = [
            i for i in range(n)
            if arr[i, v] == bnm.EndpointMark.ARROW
            and arr[v, i] == bnm.EndpointMark.TAIL
        ]
        if not parents:
            data[names[v]] = rng.standard_normal(n_samples)
        else:
            weights = rng.uniform(0.5, 1.5, size=len(parents))
            x = rng.normal(0, stdev, size=n_samples)
            for p, w in zip(parents, weights):
                x = x + w * data[names[p]].values
            data[names[v]] = x
    return data


# ---- adjacency-matrix helpers ---------------------------------------

def from_01_adj(adj, var_names=None):
    """Convert a {0,1} adjacency matrix (1 = directed edge i→j) to a
    bnm.GraphLike. If both adj[i,j] == 1 and adj[j,i] == 1, the edge
    is treated as undirected (CPDAG-style)."""
    a = np.asarray(adj)
    n = a.shape[0]
    endpoints = np.zeros((n, n), dtype=np.int8)
    for i in range(n):
        for j in range(i + 1, n):
            ij, ji = int(a[i, j]), int(a[j, i])
            if ij == 1 and ji == 1:
                endpoints[i, j] = endpoints[j, i] = bnm.EndpointMark.TAIL
            elif ij == 1:
                endpoints[i, j] = bnm.EndpointMark.ARROW
                endpoints[j, i] = bnm.EndpointMark.TAIL
            elif ji == 1:
                endpoints[i, j] = bnm.EndpointMark.TAIL
                endpoints[j, i] = bnm.EndpointMark.ARROW
    names = tuple(var_names) if var_names is not None else None
    return bnm.to_graphlike(endpoints, var_names=names)


def dag_to_cpdag(g):
    """Convert a DAG to its CPDAG (without Meek-rule closure — same
    semantics as 0.1.x's `dag_to_cpdag`). For a Meek-closed CPDAG,
    use cbcd's `DAG.to_cpdag()` instead.
    """
    g = bnm.to_graphlike(g)
    n = g.n_vars
    arr = np.array(g.endpoints)
    in_collider = np.zeros((n, n), dtype=bool)
    for v in range(n):
        parents = [
            u for u in range(n)
            if arr[u, v] == bnm.EndpointMark.ARROW
            and arr[v, u] == bnm.EndpointMark.TAIL
        ]
        for ip in range(len(parents)):
            for jp in range(ip + 1, len(parents)):
                u, w = parents[ip], parents[jp]
                if (
                    arr[u, w] == bnm.EndpointMark.NO_EDGE
                    and arr[w, u] == bnm.EndpointMark.NO_EDGE
                ):
                    in_collider[u, v] = True
                    in_collider[w, v] = True
    cpdag = np.zeros((n, n), dtype=np.int8)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if (
                arr[i, j] == bnm.EndpointMark.ARROW
                and arr[j, i] == bnm.EndpointMark.TAIL
            ):
                if in_collider[i, j]:
                    cpdag[i, j] = bnm.EndpointMark.ARROW
                    cpdag[j, i] = bnm.EndpointMark.TAIL
                else:
                    cpdag[i, j] = bnm.EndpointMark.TAIL
                    cpdag[j, i] = bnm.EndpointMark.TAIL
    return bnm.to_graphlike(cpdag, var_names=g.var_names)


# ---- perturbation helper for "fake learned graph" demos ----------

def perturb(g, n_drops=0, n_adds=0, n_reverses=0, seed=None):
    """Return a perturbed copy of g: drop / add / reverse the requested
    number of directed edges. Used as a stand-in for a real causal-
    discovery output when we don't want to depend on a PC algorithm.

    For real workflows, swap this for `cbcd.pc(data, ...)` or any
    PC implementation whose output you can wrap in `from_01_adj()`.
    """
    g = bnm.to_graphlike(g)
    rng = random.Random(seed)
    arr = np.array(g.endpoints).copy()
    n = arr.shape[0]
    directed_edges = [
        (i, j) for i in range(n) for j in range(n)
        if arr[i, j] == bnm.EndpointMark.ARROW
        and arr[j, i] == bnm.EndpointMark.TAIL
    ]
    rng.shuffle(directed_edges)
    # Drops
    for k in range(min(n_drops, len(directed_edges))):
        i, j = directed_edges[k]
        arr[i, j] = arr[j, i] = bnm.EndpointMark.NO_EDGE
    remaining = directed_edges[n_drops:]
    # Reverses
    for k in range(min(n_reverses, len(remaining))):
        i, j = remaining[k]
        arr[i, j] = bnm.EndpointMark.TAIL
        arr[j, i] = bnm.EndpointMark.ARROW
    # Adds (only between currently-non-adjacent pairs to keep it a DAG-ish)
    non_edges = [
        (i, j) for i in range(n) for j in range(i + 1, n)
        if arr[i, j] == bnm.EndpointMark.NO_EDGE
    ]
    rng.shuffle(non_edges)
    for k in range(min(n_adds, len(non_edges))):
        i, j = non_edges[k]
        # Random direction
        if rng.random() < 0.5:
            arr[i, j] = bnm.EndpointMark.ARROW
            arr[j, i] = bnm.EndpointMark.TAIL
        else:
            arr[i, j] = bnm.EndpointMark.TAIL
            arr[j, i] = bnm.EndpointMark.ARROW
    return bnm.to_graphlike(arr, var_names=g.var_names)


# ---- viz helper: 0.1.x's compare_two_bn(option=2) ----------------

def mb_pair(g1, g2, var):
    """Return ``(g1.MB(var), g2 restricted to g1.MB(var) indices)``.
    Useful for side-by-side rendering of "true MB" vs "estimated MB
    over the same nodes." Replaces 0.1.x's ``compare_two_bn(option=2)``."""
    g1n, g2n = bnm.to_graphlike(g1), bnm.to_graphlike(g2)
    idx = bnm.markov_blanket_indices(g1n, var)
    sub_g1 = bnm.markov_blanket(g1n, var)
    arr2 = g2n.endpoints[np.ix_(idx, idx)]
    sub_names = (
        tuple(g2n.var_names[i] for i in idx) if g2n.var_names else None
    )
    sub_g2 = bnm.to_graphlike(arr2, var_names=sub_names)
    return sub_g1, sub_g2


print("bnm:", bnm.__version__)


bnm: 0.2.0.dev0


# Introduction

SID quantifies how many intervention pairs (i, j) are mis-classified between a true DAG and an estimated DAG/CPDAG. Unlike SHD, SID weighs *causal-inference* consequences of structural errors — a missing direct edge that's covered by an indirect path costs less than a reversed edge.

Reference: Peters & Bühlmann (2015), https://doi.org/10.48550/arXiv.1306.1043

## Use case 1 — SID vs SHD on small examples

We compare a true DAG **G** against two estimates that both differ from G by exactly one edge (SHD = 1) but in structurally different ways:

- **H₁**: G with the X1→X2 edge replaced by an undirected   X1—X2 (a CPDAG). The Markov-equivalence class contains   G itself, so SID's lower bound is 0.
- **H₂**: G with the X1→X2 edge reversed (X2→X1). A pure   DAG that differs causally; SID > 0.

In [2]:
nodes = ['X1', 'X2', 'Y1', 'Y2', 'Y3']

G = np.array([
    [0, 1, 1, 1, 1],
    [0, 0, 1, 1, 1],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
])
# H1: X1—X2 undirected (a CPDAG); rest identical to G.
H1 = np.array([
    [0, 1, 1, 1, 1],
    [1, 0, 1, 1, 1],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
])
# H2: X1→X2 reversed to X2→X1; everything else identical.
H2 = np.array([
    [0, 0, 1, 1, 1],
    [1, 0, 1, 1, 1],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
])

g_true = from_01_adj(G, var_names=nodes)
g_h1 = from_01_adj(H1, var_names=nodes)
g_h2 = from_01_adj(H2, var_names=nodes)

### Case 1 — G vs H₁ (CPDAG with G in its equivalence class)

In [3]:
bnm.plot_side_by_side(
    g_true, g_h1, name1='G', name2='H1', direction='auto'
)

In [4]:
c = bnm.compare(g_true, g_h1, comparative=['shd'], include_sid=True)
print(f'SHD = {c.comparative["shd"]}, '
      f'SID = {c.sid.sid}, '
      f'bounds = [{c.sid.sid_lower_bound}, {c.sid.sid_upper_bound}], '
      f'is_tight = {c.sid.is_tight}')

SHD = 1.0, SID = 8, bounds = [0, 8], is_tight = False


**Key observations**:

- SHD(G, H₁) = 1 (one orientation differs).
- SID lower bound = 0 — when X1—X2 is oriented as X1→X2 (picking G itself out of the equivalence class), every intervention prediction matches G.
- SID upper bound > 0 — the alternative orientation (X2→X1) does change interventions and gives a non-zero SID.
- `is_tight = False` flags that the bounds aren't equal — the equivalence class is non-trivial.

The `c.sid.sid` value itself counts (i, j) pairs that are mis-classified by **at least one** DAG in the class — useful when you want a worst-case-over-class estimate.

In [5]:
bnm.plot_sid_matrix(c.sid)

**Conclusion**: SID has a `[lower, upper]` interval when the estimate is a CPDAG. The lower bound = 0 here means *there exists* a DAG in H₁'s class that perfectly matches G's intervention predictions — the orientation ambiguity doesn't force any wrong call.

### Case 2 — G vs H₂ (single reversed edge)

In [6]:
c = bnm.compare(g_true, g_h2, comparative=['shd'], include_sid=True)
print(f'SHD = {c.comparative["shd"]}, '
      f'SID = {c.sid.sid}, '
      f'bounds = [{c.sid.sid_lower_bound}, {c.sid.sid_upper_bound}], '
      f'is_tight = {c.sid.is_tight}')

SHD = 1.0, SID = 8, bounds = [8, 8], is_tight = True


In [7]:
bnm.plot_sid_matrix(c.sid)

**Key observations**:

- SHD(G, H₂) = 1 — same as G vs H₁.
- SID > 0 — the reversed edge creates real intervention differences.
- Bounds are tight (lower == upper) — H₂ is a pure DAG, no equivalence-class ambiguity.

**Takeaway**: two graphs can have equal SHD but very different SID. SHD is a structural distance; SID is a causal-inference distance.

## Use case 2 — DAG bounds via CPDAG ambiguity

When the estimated graph is a CPDAG, the orientation of undirected edges is ambiguous. SID has a `[lower, upper]` interval reflecting the min and max SID over all DAGs in the equivalence class.

In [8]:
nodes4 = ['X1', 'X2', 'X3', 'X4']
DAG_a = np.array([
    [0, 0, 0, 0],
    [0, 0, 1, 0],
    [1, 0, 0, 0],
    [1, 0, 1, 0],
])
DAG_b = np.array([
    [0, 1, 0, 0],
    [0, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
])

g_dag_a = from_01_adj(DAG_a, var_names=nodes4)
g_dag_b = from_01_adj(DAG_b, var_names=nodes4)

for label, g_est in [('DAG_b', g_dag_b)]:
    c = bnm.compare(g_dag_a, g_est, comparative=['shd'], include_sid=True)
    print(f'{label}: SHD={c.comparative["shd"]}, SID={c.sid.sid}, bounds=[{c.sid.sid_lower_bound}, {c.sid.sid_upper_bound}]')

DAG_b: SHD=4.0, SID=7, bounds=[7, 7]


Comparing the same true DAG to its own CPDAG (the Markov-equivalence class):

In [9]:
cpdag_a = dag_to_cpdag(g_dag_a)
c = bnm.compare(g_dag_a, cpdag_a, comparative=['shd'], include_sid=True)
print(f'truth vs CPDAG-of-truth: SHD={c.comparative["shd"]}, SID={c.sid.sid}, bounds=[{c.sid.sid_lower_bound}, {c.sid.sid_upper_bound}]')
print(f'is_tight: {c.sid.is_tight}')

truth vs CPDAG-of-truth: SHD=2.0, SID=9, bounds=[6, 9]
is_tight: False


`is_tight=False` indicates the bound interval is non-trivial — the equivalence class contains DAGs that yield different SID values against the truth.

## Use case 3 — SID on Markov blankets (local)

We can localise SID to a node's Markov blanket: build the MB sub-graphs of g1 and g2 over the same node indices, compute SID on the pair.

We use a perturbation as a stand-in for a real PC output. For real workflows, swap `perturb(...)` for `cbcd.pc(data, ...)`.

In [10]:
true_dag = random_dag(n_nodes=40, edge_prob=0.1, seed=55)
learned = perturb(true_dag, n_drops=12, n_reverses=8, n_adds=6, seed=42)

### Local SID for a single variable

In [11]:
var = 'X_16'
sub1, sub2 = mb_pair(true_dag, learned, var)

bnm.plot_side_by_side(
    sub1, sub2,
    name1=f'True MB of {var}',
    name2=f'Learned over the same nodes',
    highlight_nodes=[var],
    direction='auto',
)

In [12]:
# Note: SID requires g1 to be a pure DAG. The MB sub-graph
# of a DAG is itself a DAG; the learned MB sub-graph is
# possibly a CPDAG (TAIL/TAIL undirected edges).
sid_local = bnm.sid(sub1, sub2)
print(f'local SID for {var}: sid={sid_local.sid}, bounds=[{sid_local.sid_lower_bound}, {sid_local.sid_upper_bound}]')

bnm.plot_sid_matrix(sid_local)

local SID for X_16: sid=27, bounds=[27, 27]


### Local SID for multiple variables

Using the union of MB index sets gives a sub-graph containing both anchor variables and their dependencies.

In [13]:
def union_mb(g, vars_):
    g_norm = bnm.to_graphlike(g)
    indices = sorted({i for v in vars_
                      for i in bnm.markov_blanket_indices(g_norm, v)})
    arr = g_norm.endpoints[np.ix_(indices, indices)]
    names = (
        tuple(g_norm.var_names[i] for i in indices)
        if g_norm.var_names else None
    )
    return bnm.to_graphlike(arr, var_names=names), indices

vars_ = ['X_16', 'X_35']
sub_truth, idx = union_mb(true_dag, vars_)

# Restrict learned to the same indices.
learned_n = bnm.to_graphlike(learned)
sub_learned = bnm.to_graphlike(
    learned_n.endpoints[np.ix_(idx, idx)],
    var_names=tuple(learned_n.var_names[i] for i in idx)
    if learned_n.var_names else None,
)

bnm.plot_side_by_side(
    sub_truth, sub_learned,
    name1=f'True MBs of {vars_}',
    name2='Learned over the same nodes',
    highlight_nodes=vars_,
    direction='auto',
)

In [14]:
sid_multi = bnm.sid(sub_truth, sub_learned)
print(f'union-MB SID: {sid_multi.sid}')
bnm.plot_sid_matrix(sid_multi)

union-MB SID: 44


## Conclusion

Three SID workflows in v0.2:

1. **Whole-graph SID**: `bnm.sid(g_true, g_est)` returns a `SIDResult` with `sid`, `sid_lower_bound`, `sid_upper_bound`, and an `incorrect_mat` (n × n) for heatmap rendering.
2. **CPDAG bounds**: when `g_est` is a CPDAG, the bounds interval reflects equivalence-class ambiguity. `SIDResult.is_tight` flags whether `lower == upper`.
3. **Local SID**: build MB sub-graphs (over the same node indices via `mb_pair` / `union_mb`) and compute SID on the pair.

All viz functions (`plot_side_by_side`, `plot_sid_matrix`) accept `save=path` to write to disk.

### Other Cases:
- [Evaluate Single DAG](./evaluate_single_dag.ipynb)
- [Compare Two DAGs](./compare_two_dags.ipynb)
- [Compare Algorithms](./compare_algorithms.ipynb)